# Python on Viya — release smoke test (notebook)

Not part of any automated suite. Pick **Python on Viya** in the kernel picker, then **Run All**.

- Each check prints `PASS` / `FAIL` / `SKIP`; `LOOK` lines name something to confirm by eye in the cell output.
- The **Summary** cell raises if any check failed.
- The two cells after the summary are manual checks (interrupt, traceback). Run All reaches them too: the interrupt cell takes about 20 seconds unless you stop it, and the traceback cell **fails on purpose**.
- Run `release_smoke.py` first if you want the "own compute session" check to mean something.

**Clear All Outputs before committing this file** — saved outputs can carry deployment details.

In [ ]:
# `SAS` is injected into the interpreter by PROC PYTHON, so Pylance cannot see it.
# pyright: reportUndefinedVariable=false
import os
import sys
import time

_results = {}  # name -> status; re-running a cell overwrites its entries


class Skip(Exception):
    """Raised inside a check to report SKIP rather than FAIL."""


def expect(condition, message):
    if not condition:
        raise AssertionError(message)


def check(name):
    """Decorator that runs the function immediately and records its outcome."""

    def run(fn):
        try:
            detail = fn()
            status = "PASS"
        except Skip as exc:
            status, detail = "SKIP", str(exc)
        except Exception as exc:
            status, detail = "FAIL", f"{type(exc).__name__}: {exc}"
        _results[name] = status
        print(f"[{status}] {name}" + (f" -- {detail}" if detail else ""), flush=True)
        return fn

    return run


def look(message):
    print(f"[LOOK] {message}", flush=True)


CAPTURE_FILES = ["smoke_nb_01_plot.png", "smoke_nb_02_table.html", "smoke_nb_03_sanitizer.html"]
print("=== Python on Viya release smoke test (notebook) ===")

In [ ]:
@check("SAS bridge object is present")
def _():
    expect("SAS" in globals(), "no SAS object: this is not running under PROC PYTHON")
    return f"Python {sys.version.split()[0]}"


@check("Notebook has its own compute session")
def _():
    # release_smoke.py sets this in the Run File session; it must not be visible here.
    expect("_SMOKE_PY_SENTINEL" not in globals(), "the Run File session's namespace is visible")
    return "meaningful only if release_smoke.py ran first"


@check("No leftover capture files from a previous run")
def _():
    leftovers = [name for name in CAPTURE_FILES if os.path.exists(name)]
    expect(not leftovers, f"still present: {', '.join(leftovers)} (fine if a cell was interrupted)")


_nb_state_marker = "set in an earlier cell"

In [ ]:
@check("State carries over between cells")
def _():
    expect(globals().get("_nb_state_marker") == "set in an earlier cell", "variable from the cell above is gone")


@check("stdout arrives in the cell complete and in order")
def _():
    for i in range(1, 6):
        print(f"    printed line {i}/5", flush=True)
        time.sleep(0.5)


look("the five 'printed line' lines in order 1-5, all appearing when the cell finished (Finding 13.3)")

In [ ]:
_class_df = None


@check("SAS.sd2df reads sashelp.class")
def _():
    global _class_df
    _class_df = SAS.sd2df("sashelp.class")
    expect(_class_df.shape == (19, 5), f"shape {_class_df.shape}, expected (19, 5)")


@check("SAS.df2sd round trip through work.smoke_nb_class")
def _():
    if _class_df is None:
        raise Skip("sd2df failed above")
    SAS.df2sd(_class_df, "work.smoke_nb_class")
    back = SAS.sd2df("work.smoke_nb_class")
    expect(len(back) == 19, f"{len(back)} rows read back")


@check("SAS.submit PROC SQL view, then sd2df of the view")
def _():
    SAS.submit(
        """
proc sql;
  create view work.smoke_nb_boys as
    select name, age from sashelp.class where sex = 'M';
quit;
"""
    )
    boys = SAS.sd2df("work.smoke_nb_boys")
    expect(len(boys) == 10, f"{len(boys)} rows, expected 10")


@check("SAS.symput / SAS.symget with non-ASCII text")
def _():
    SAS.symput("smoke_nb_mv", "café ✓")
    value = SAS.symget("smoke_nb_mv")
    expect(value == "café ✓", repr(value))


@check("endsubmit; inside a string survives unchanged")
def _():
    trap = """
endsubmit;
run;
"""
    expect(trap == "\n" + "end" + "submit;\nrun;\n", repr(trap))

In [ ]:
@check("SAS.show(df) and SAS.show(plt, filetype='png')")
def _():
    if not hasattr(SAS, "show"):
        raise Skip("SAS.show needs Viya 2025.03 or later")
    if _class_df is None:
        raise Skip("sd2df failed above")
    SAS.show(_class_df.head(5))
    try:
        import matplotlib

        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
    except ImportError:
        raise Skip("matplotlib is not installed on the deployment")
    plt.figure()
    plt.plot([1, 2, 3, 4], [3, 1, 4, 2], marker="o")
    plt.title("smoke nb: SAS.show png")
    SAS.show(plt, filetype="png")
    plt.close("all")


look("a 5-row table in the notebook's own table styling, then a line chart 'smoke nb: SAS.show png'")

In [ ]:
@check("SAS.show(plt) without filetype (SVG)")
def _():
    if not hasattr(SAS, "show"):
        raise Skip("SAS.show needs Viya 2025.03 or later")
    try:
        import matplotlib

        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
    except ImportError:
        raise Skip("matplotlib is not installed on the deployment")
    plt.figure()
    plt.plot([1, 2, 3], [1, 3, 2])
    SAS.show(plt)
    plt.close("all")


look("a one-line note in place of the SVG figure (notebooks do not render SAS.show SVG)")

In [ ]:
@check("SAS.submit PROC SGPLOT and PROC PRINT")
def _():
    # SGPLOT first on purpose: on v0.1.4, as the last submitted step it hides
    # the cell's printed output (release_smoke_sgplot.py).
    SAS.submit(
        """
title "smoke nb: proc sgplot";
proc sgplot data=sashelp.class; vbar sex / response=height stat=mean; run;
title "smoke nb: proc print";
proc print data=sashelp.class(obs=3); run;
title;
"""
    )


look("a 'smoke nb: proc sgplot' bar chart image and a 3-row 'smoke nb: proc print' table")

In [ ]:
@check("Written .png and .html files are captured")
def _():
    try:
        import matplotlib

        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
    except ImportError:
        raise Skip("matplotlib is not installed on the deployment")
    fig, ax = plt.subplots()
    ax.bar(["a", "b", "c"], [2, 5, 3])
    ax.set_title("smoke nb: savefig png")
    fig.savefig(CAPTURE_FILES[0])
    plt.close(fig)
    if _class_df is not None:
        _class_df.head(3).to_html(CAPTURE_FILES[1])


@check("HTML with a <script> is written for the sanitizer check")
def _():
    html = (
        "<h3>smoke nb: sanitizer</h3>"
        '<p id="smoke-target">SAFE: this sentence must be unchanged.</p>'
        "<script>document.getElementById('smoke-target').textContent='SCRIPT RAN - FAIL';</script>"
        '<img src="x" alt="" onerror="document.body.style.background=\'red\'">'
    )
    with open(CAPTURE_FILES[2], "w", encoding="utf-8") as handle:
        handle.write(html)


look("'smoke nb: savefig png' bar chart, a 3-row HTML table, and 'SAFE: ...' unchanged with nothing red")

## Summary

In [ ]:
counts = {s: sum(1 for v in _results.values() if v == s) for s in ("PASS", "FAIL", "SKIP")}
print(f"PASS {counts['PASS']}   FAIL {counts['FAIL']}   SKIP {counts['SKIP']}")
for name, status in _results.items():
    if status != "PASS":
        print(f"  {status}: {name}")
if counts["FAIL"]:
    raise AssertionError(f"{counts['FAIL']} smoke check(s) failed; see [FAIL] lines above")

## Manual checks

**Interrupt.** Run the next cell and press the cell's stop button a few seconds in. Its countdown lines do not show while it runs, because printed output arrives only when a cell ends (Finding 13.3), so count the seconds yourself. The cell stops at once. Then run the traceback cell straight away: after a few silent seconds it should show a notice that SAS Viya may still be finishing the interrupted statement, then run.

**Traceback.** The last cell fails on purpose with `ZeroDivisionError`. Expect the traceback in the cell output and one **Problems** entry on the `return numerator / denominator` line of that cell. Running the cell again (still failing) replaces the entry rather than adding one; closing the notebook clears it.

In [ ]:
for second in range(1, 21):
    print(f"interrupt demo: {second}/20 s", flush=True)
    time.sleep(1)
print("interrupt demo finished without being stopped")

In [ ]:
def outer():
    return inner(10, 0)


def inner(numerator, denominator):
    return numerator / denominator


outer()